In [112]:
import pandas as pd
import os
import numpy as np  # <--- เพิ่มบรรทัดนี้ครับ

In [113]:
Month = 1
session = 'u' # /// g คือชุดทั่วไป /// u คือชุดนอกเขต
Year = 2569

Web_Data_SsoTpso = 1 # /// อยากให้รันข้อมูลสินค้นให้กดหนึ่ง /// ถ้าไม่รันกดศูนย์
Web_Shop_12C = 0
text = session

In [114]:
masterF = 'Real_Master_CPI.xlsx'
# Ensure the file exists before reading
try:
    Real_master = pd.read_excel(masterF)
    RM = Real_master[['รหัส', 'รายการ', 'ผู้ดูแล', 'กำหนดให้เก็บ', 'รายละเอียด','G/L','U']].rename(columns={'รหัส': 'CODE7'})
    RM["CODE7"] = RM["CODE7"].astype(str).str.zfill(7)
    RM = RM.dropna(subset=['กำหนดให้เก็บ'])
    RM['กำหนดให้เก็บ'] = pd.to_numeric(RM['กำหนดให้เก็บ'], errors='coerce')
    RM = RM[RM['กำหนดให้เก็บ'] > 0]    
    RM['กำหนดให้เก็บ'] = RM['กำหนดให้เก็บ'].astype(int)

    if text =='g':
        RM = RM[RM['G/L']>0]
    elif text == 'u':
        RM = RM[RM['U']>0]
    RM = RM[['CODE7', 'รายการ', 'ผู้ดูแล', 'กำหนดให้เก็บ', 'รายละเอียด']]
    rm_gits = RM['CODE7'].tolist()  

    RM = RM.reset_index(drop=True)
    # print(RM.head())

except FileNotFoundError:
    print(f"File {masterF} not found.")

rm_gits = RM['CODE7'].tolist()  

# RM
len(set(rm_gits))

403

In [115]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import os, time

if Web_Data_SsoTpso==1:
    # ===== ฟังก์ชันรอจนไฟล์โหลดเสร็จ =====
    def wait_for_downloads(folder_path, timeout=120):
        """รอจนไม่มีไฟล์ .crdownload ในโฟลเดอร์"""
        start_time = time.time()
        while True:
            # ถ้ายังมีไฟล์ .crdownload แสดงว่ายังโหลดไม่เสร็จ
            downloading = any(f.endswith('.crdownload') for f in os.listdir(folder_path))
            if not downloading:
                time.sleep(15)
                break
            if time.time() - start_time > timeout:
                raise TimeoutError("⏰ โหลดไฟล์ไม่เสร็จภายในเวลาที่กำหนด")
            time.sleep(1)

    # ===== ตั้งค่า Chrome ให้โหลดไฟล์มาที่โฟลเดอร์ปัจจุบัน =====
    download_dir = os.getcwd()
    chrome_options = webdriver.ChromeOptions()
    prefs = {
        "download.default_directory": download_dir,
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "safebrowsing.enabled": True
    }
    chrome_options.add_experimental_option("prefs", prefs)

    # ===== ลบไฟล์ Excel เดิมทั้งหมดก่อนเริ่มดาวน์โหลด =====
    for f in os.listdir(download_dir):
        # เช็คชื่อหน้า: ขึ้นต้นด้วย "cpi" หรือ "Unconfirmed"
        # เช็คชื่อหลัง: ลงท้ายด้วย ".xlsx" หรือ ".tmp" หรือ ".crdownload"
        if f.startswith(("cpi","เงินเดือน","ราคาสินค้า", "Unconfirmed")) and f.endswith((".xlsx", ".tmp", ".crdownload")):
            try:
                os.remove(os.path.join(download_dir, f))
                print(f"ลบไฟล์เก่า: {f}")
            except Exception as e:
                print(f"ไม่สามารถลบ {f}: {e}")

    # ===== เริ่มเปิดเบราว์เซอร์ =====
    driver = webdriver.Chrome(options=chrome_options)
    driver.get("https://sso-tpso.moc.go.th/Login?ReturnUrl=%2F")

    # ===== เข้าระบบ =====
    driver.find_element(By.ID, "Username").send_keys("tanaponp")
    driver.find_element(By.ID, "Password").send_keys("&5#t2B3&")
    driver.find_element(By.ID, "Password").send_keys(Keys.RETURN)
    time.sleep(2)

    # ===== ไปหน้าที่มีปุ่ม cpig month =====
    driver.get("https://dev-tpso.moc.go.th/export12c/cpi")
    time.sleep(2)

    driver.execute_script(f"document.getElementById('ddlMonth').value = '{Month}';")
    driver.execute_script(f"document.getElementById('ddlYear').value = '{Year}';")

    # ===== คลิกปุ่ม cpig month =====
    if session == 'g':
        driver.find_element(By.ID, "btnCpigMonth").click()
        print("📂 กำลังดาวน์โหลดไฟล์ cpig_month_2568-10.xlsx ...")
    elif session == 'u':
        driver.find_element(By.ID, "btnCpiuMonth").click()
        print("📂 กำลังดาวน์โหลดไฟล์ cpig_month_2568-10.xlsx ...")        


    # ✅ แทน time.sleep(50)
    # wait_for_downloads(download_dir, timeout=180)
    time.sleep(35)

        # ===== คลิกปุ่ม cpig month =====
    if session == 'g':
        driver.find_element(By.ID, "btnCpigWeek").click()
        print("📂 กำลังดาวน์โหลดไฟล์ cpig_month_2568-10.xlsx ...")
    elif session == 'u':
        driver.find_element(By.ID, "btnCpiuWeek").click()
        print("📂 กำลังดาวน์โหลดไฟล์ cpig_month_2568-10.xlsx ...")        


    # ✅ แทน time.sleep(50)
    # wait_for_downloads(download_dir, timeout=180)
    time.sleep(35)

    # ===== ตรวจสอบว่ามีไฟล์อยู่ในโฟลเดอร์ =====
    files = [f for f in os.listdir(download_dir) if f.endswith(".xlsx")]
    print("✅ ไฟล์ที่โหลดแล้ว:", files)

    driver.quit()




ลบไฟล์เก่า: cpiu_month_2569-2.xlsx
ลบไฟล์เก่า: cpiu_week_2569-2.xlsx
📂 กำลังดาวน์โหลดไฟล์ cpig_month_2568-10.xlsx ...
📂 กำลังดาวน์โหลดไฟล์ cpig_month_2568-10.xlsx ...
✅ ไฟล์ที่โหลดแล้ว: ['cpiu_month_2569-1.xlsx', 'cpiu_week_2569-1.xlsx', 'Real_Master_CPI.xlsx', 'shop_data_20260305_100619.xlsx', 'useCase_nongkhai.xlsx', 'useCase_puket.xlsx', 'useCase_puket_grouped.xlsx', 'useCase_puket_styled.xlsx', 'useCase_puket_เมือง.xlsx', 'การเก็บวิเคราะห์การเก็บ โดยละเอียด ชุดทั่วไป.xlsx', 'การเก็บวิเคราะห์การเก็บ โดยละเอียด ชุดนอกเขต.xlsx', 'จังหวัดใช้งาน.xlsx']


In [116]:
if (Web_Data_SsoTpso==1) & (session == 'g') & (Web_Shop_12C == 10):
    from selenium import webdriver
    from selenium.webdriver.common.by import By
    from selenium.webdriver.common.keys import Keys
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC
    import os, time


    # ===== ตั้งค่า Chrome ให้โหลดไฟล์ลง workspace ปัจจุบัน =====
    download_dir = os.getcwd()
    chrome_options = webdriver.ChromeOptions()
    prefs = {
        "download.default_directory": download_dir,
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "safebrowsing.enabled": True
    }
    chrome_options.add_experimental_option("prefs", prefs)

    # ===== ลบไฟล์ เดิมทั้งหมดก่อนเริ่มดาวน์โหลด =====
    for f in os.listdir(download_dir):
        if f.startswith("shop_data") & f.endswith(".xlsx"):
            try:
                os.remove(os.path.join(download_dir, f))
                print(f"ลบไฟล์เก่า: {f}")
            except Exception as e:
                print(f"ไม่สามารถลบ {f}: {e}")

    driver = webdriver.Chrome(options=chrome_options)

    # ===== เปิดหน้า login =====
    driver.get("https://cpi.moc.go.th/cpi/login.aspx")

    # ===== รอจนฟิลด์ username โหลดขึ้นมา =====
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.ID, "txtUser"))
    )

    # ===== กรอก username / password =====
    driver.find_element(By.ID, "txtUser").send_keys("index_bkk027")
    driver.find_element(By.ID, "txtPwd").send_keys("Tpsocpi1234")

    # ===== คลิกปุ่ม login =====
    driver.find_element(By.ID, "btnLogin").click()

    # ===== รอจนเข้าสู่ระบบสำเร็จ (หน้า login จะ redirect อัตโนมัติ) =====
    # ตรวจว่าหลุดจากหน้า login แล้ว
    WebDriverWait(driver, 20).until_not(
        EC.url_contains("login.aspx")
    )

    # ===== เข้าไปหน้าจัดการร้านค้า =====
    driver.get("https://cpi.moc.go.th/cpi/shopMgt.aspx")

    # ===== รอให้ปุ่ม Excel ปรากฏ =====
    WebDriverWait(driver, 15).until(
        EC.element_to_be_clickable((By.ID, "ContentPlaceHolder1_btnExcel"))
    )

    # ===== คลิกปุ่มเพื่อดาวน์โหลดไฟล์ Excel =====
    driver.find_element(By.ID, "ContentPlaceHolder1_btnExcel").click()

    print("📂 กำลังดาวน์โหลดไฟล์ร้านกรุงเทพ...")

    # ===== รอให้ไฟล์โหลดเสร็จ ===== 
    time.sleep(15)

    # ===== ตรวจสอบว่ามีไฟล์อยู่ในโฟลเดอร์ =====
    files = [f for f in os.listdir(download_dir) if f.endswith(".xlsx")]
    print("ไฟล์ที่โหลดแล้ว:", files)

    driver.quit()


In [117]:
# if text =='g':
#     RM = RM[RM['G/L']>0]
# elif text == 'u':
#     RM = RM[RM['U']>0]
# RM = RM[['CODE7', 'รายการ', 'ผู้ดูแล', 'กำหนดให้เก็บ', 'รายละเอียด']]
# rm_gits = RM['CODE7'].tolist()  

# RM
# len(set(rm_gits))
# RM

In [118]:
if session == 'g':
    mg_ = pd.read_excel("cpi"+str(session)+"_month_"+str(Year)+"-"+str(Month)+".xlsx", engine='calamine')
    wg_ = pd.read_excel("cpi"+str(session)+"_week_"+str(Year)+"-"+str(Month)+".xlsx", engine='calamine')
elif session == 'u':  
    mu_ = pd.read_excel("cpi"+str(session)+"_month_"+str(Year)+"-"+str(Month)+".xlsx", engine='calamine')
    wu_ = pd.read_excel("cpi"+str(session)+"_week_"+str(Year)+"-"+str(Month)+".xlsx", engine='calamine')

# Real_master = pd.read_excel(masterF)
# if run_cpi_month == 1:
#     # Main_dataM = pd.read_excel("cpi"+str(session)+"_month_"+str(Year)+"-"+str(Month)+".xlsx", engine='calamine')
#     # Main_dataW = pd.read_excel("cpi"+str(session)+"_week_"+str(Year)+"-"+str(Month)+".xlsx", engine='calamine')    

In [119]:
if session == 'g':
    mg = mg_.copy()
    wg = wg_.copy()
    mg.insert(0, 'ชุด', 'อ.เมือง')    
    wg.insert(0, 'ชุด', 'อ.เมือง')
elif session == 'u':  
    mu = mu_.copy()
    wu = wu_.copy()
    mu.insert(0, 'ชุด', 'นอกเขต')
    wu.insert(0, 'ชุด', 'นอกเขต')


# --- ต่อจาก Code ของคุณ ---

# 1. เลือกว่าจะใช้ชุดข้อมูลไหนตรวจสอบ (ตามตัวแปร text)
if text == 'g':
    # รวมข้อมูล General (รายเดือน + รายสัปดาห์)
    df_actual = pd.concat([mg, wg], ignore_index=True)

# คำนวณ Flag

    target_set_name = 'ชุดทั่วไป'
elif text == 'u':
    # รวมข้อมูล Rural/Urban (รายเดือน + รายสัปดาห์)
    df_actual = pd.concat([mu, wu], ignore_index=True)
    target_set_name = 'ชุดนอกเขต'
else:
    raise ValueError("ตัวแปร text ต้องเป็น 'g' หรือ 'u' เท่านั้น")

df = df_actual.copy()

# จัด Format รหัส
df["COMMODITY_CODE"] = df["COMMODITY_CODE"].astype(str).str.zfill(16)
df["DILLER_CODE"]    = df["DILLER_CODE"].astype(str).str.zfill(10)
df["CODE7"]  = df["COMMODITY_CODE"].astype(str).str[:7]
df["CODE7"] = df["COMMODITY_CODE"].str[0:7]
df["CODE9"] = df["COMMODITY_CODE"].str[7:16]
df.sort_values(by=['COMMODITY_CODE'], inplace=True)
df.reset_index(drop=True)

import pandas as pd
import glob
# ค้นหาไฟล์ทั้งหมดที่ขึ้นต้นด้วย shop_data และลงท้ายด้วย .xlsx
files_shop = glob.glob("shop_data*.xlsx")
ds_ = pd.read_excel(files_shop [0], skiprows=2, engine='calamine')
# Dealer Group
ds = ds_[["รหัสร้าน", "กลุ่ม"]].rename(columns={"รหัสร้าน": "DILLER_CODE"})
ds["DILLER_CODE"] = ds["DILLER_CODE"].astype(str).str.zfill(10)
ds["กลุ่ม"] = ds["กลุ่ม"].replace(["-", " "], "", regex=True)
df = df.merge(ds, on=["DILLER_CODE"], how="left")

df.loc[df['PROVINCE_NAME'] != 'กรุงเทพมหานคร', 'GROUP'] = df['PROVINCE_NAME']
df.loc[df['PROVINCE_NAME'] == 'กรุงเทพมหานคร', 'GROUP'] = 'กทม.'+df['กลุ่ม']
df = df.reindex(columns=[ 'CODE7', 'GROUP'])
df.to_clipboard()


In [120]:
df

,CODE7,GROUP
0,1111001,กระบี่
1,1111001,สงขลา
2,1111001,กระบี่
3,1111001,นครศรีธรรมราช
4,1111001,พิษณุโลก
...,...,...
38284,7230001,ปราจีนบุรี
38285,7230001,เชียงใหม่
38286,7230001,ชุมพร
38287,7230001,ราชบุรี


In [121]:
Detail = RM[['CODE7','รายการ','กำหนดให้เก็บ','รายละเอียด']].copy()
Detail

,CODE7,รายการ,กำหนดให้เก็บ,รายละเอียด
0,1111001,ข้าวสารเจ้า,12,2แหล่ง (1หอมมะลิแบบตัก/2หอมมะลิแบบถุง/1ขาวเสาไ...
1,1111002,ข้าวสารเหนียว,4,2แหล่ง (1ข้าวเหนียวถัง/1ข้าวเหนียวถุง)
2,1112104,แป้งทอดกรอบ,4,2แหล่ง 2สเปค
3,1112202,วุ้นเส้น,8,2แหล่ง (2ขนาดเล็ก/2ขนาดใหญ่)
4,1112203,เต้าหู้,6,2แหล่ง (2เต้าหู้หลอด/1เต้าหู้แผ่นแข็ง)
...,...,...,...,...
398,6311001,เครื่องถวายพระ,8,2แหล่ง (1ดอกไม้ไหว้พระ/1ธูป/1เทียน/1ชุดผ้าไตรจ...
399,6311002,ค่าอาหารสำหรับตักบาตร,2,2แหล่ง 1สเปค
400,7100001,บุหรี่,8,2แหล่ง (2บุหรี่นอก/2บุหรี่ใน)
401,7210001,เบียร์,8,2แหล่ง (2เบียร์นอก/2เบียร์ใน)


In [122]:
province_list_exist = pd.read_excel('จังหวัดใช้งาน.xlsx', engine='calamine')

if text == 'g':
    province_list = province_list_exist['ชุดทั่วไป'].dropna().tolist()
elif text == 'u':
    province_list = province_list_exist['ชุดนอกเขต'].dropna().tolist()
text    

'u'

In [123]:
print(len(df['CODE7'].unique()))
print(len(rm_gits))
# หาตัวที่มีใน df['CODE7'] แต่ไม่มีใน rm_gits
diff = set(df['CODE7'].unique()) - set(rm_gits)
print(f"รหัสที่เกินมาคือ: {diff}")
# หาตัวที่มีใน df['CODE7'] แต่ไม่มีใน rm_gits
diff = set(set(rm_gits) - set(df['CODE7'].unique()))

print(f"รหัสที่เกินมาคือ: {diff}")

unnion_codes = set(df['CODE7'].unique()).union(set(rm_gits))
print(f"รหัสทั้งหมดที่ควรมี: {len(unnion_codes)}")

453
403
รหัสที่เกินมาคือ: {'5110001', '1152205', '4111020', '3400011', '5230005', '1132005', '4112005', '6221203', '3520014', '6130010', '1121203', '1152206', '3700001', '3150001', '5220008', '1121211', '2123010', '5420008', '2115004', '3130003', '6213002', '3510006', '1141119', '1123203', '1141133', '1142123', '1132002', '3520008', '1141130', '2132002', '1123407', '1142118', '3600016', '1141124', '1160004', '3400006', '1142122', '3140002', '5110006', '6125003', '4222002', '3510002', '5211002', '1141125', '1171002', '2113003', '1123210', '7220001', '3700003', '2212001', '2123003', '3520010', '3150002', '6110002'}
รหัสที่เกินมาคือ: {'6214002', '4121107', '6110001', '6214001'}
รหัสทั้งหมดที่ควรมี: 457


In [124]:
df

,CODE7,GROUP
0,1111001,กระบี่
1,1111001,สงขลา
2,1111001,กระบี่
3,1111001,นครศรีธรรมราช
4,1111001,พิษณุโลก
...,...,...
38284,7230001,ปราจีนบุรี
38285,7230001,เชียงใหม่
38286,7230001,ชุมพร
38287,7230001,ราชบุรี


In [125]:
import pandas as pd
import numpy as np

# ... (Previous code remains the same up to unitGROUP filtering) ...

unit7 = df['CODE7'].unique()
unitGROUP = df['GROUP'].unique()    
unitGROUP = unitGROUP[np.isin(unitGROUP, province_list)]

# --- FIX STARTS HERE ---
import pandas as pd
import numpy as np

# ... (โค้ดส่วนเตรียม unitGROUP และ all_combinations เหมือนเดิม) ...

# --- แก้ไขส่วนนี้ (FIX STARTS HERE) ---

# 1. สร้างตารางแม่แบบทุกคู่ (เหมือนเดิม)
all_combinations = pd.MultiIndex.from_product(
    [rm_gits, unitGROUP], 
    names=['CODE7', 'GROUP']
).to_frame(index=False)

# 2. คำนวณ "ค่าจริง" จาก df (เปลี่ยนจากใส่เลข 1 เป็นการนับจำนวน)
# ใช้ groupby เพื่อดูว่า รหัสนี้ในจังหวัดนี้ มีกี่แถว (Count)
# หรือถ้าต้องการ "ราคาเฉลี่ย" ให้เปลี่ยน .size() เป็น ['AVG'].mean()
actual_data = df.groupby(['CODE7', 'GROUP']).size().reset_index(name='เก็บมาได้')

# 3. Merge ข้อมูลจริงเข้ากับตารางแม่แบบ
# Left join เพื่อคงโครงสร้าง Master x Province ไว้
merged_data = all_combinations.merge(actual_data, on=['CODE7', 'GROUP'], how='left')

# เติม 0 ให้ช่องที่ไม่มีข้อมูล (คือหาไม่เจอเลย = 0)
merged_data['เก็บมาได้'] = merged_data['เก็บมาได้'].fillna(0).astype(int)

# --- ส่วนสร้าง Save 1, 2, 3 (Logic เดิม แต่ค่าข้างในจะเป็นจำนวนจริงแล้ว) ---

# Sheet 1: Matrix
crosstab_result = merged_data.pivot(index='CODE7', columns='GROUP', values='เก็บมาได้')
save1 = crosstab_result.reset_index()
save1.columns.name = None 
save1 = save1.merge(Detail, on=['CODE7'], how='left')

cols_to_move = ['รายการ', 'กำหนดให้เก็บ', 'รายละเอียด']
remaining_cols = [c for c in save1.columns if c != 'CODE7' and c not in cols_to_move]
new_column_order = ['CODE7'] + cols_to_move + remaining_cols
save1 = save1[new_column_order]

# Sheet 2: Long Format (แสดงค่าจริง)
save2 = merged_data.copy()
save2 = save2.merge(Detail, on=['CODE7'], how='left')

selected_columns = ['CODE7', 'รายการ', 'GROUP', 'เก็บมาได้', 'กำหนดให้เก็บ', 'รายละเอียด']  
available_cols = [c for c in selected_columns if c in save2.columns]
save2 = save2[available_cols]

# Sheet 3: Missing Only (เฉพาะที่ได้ 0)
save3 = save2[save2['เก็บมาได้'] == 0].copy()

# แสดงผลทดสอบ
print("ตัวอย่างข้อมูลใน Save2 (แสดงจำนวนที่เก็บได้จริง):")
display(save2.head())

ตัวอย่างข้อมูลใน Save2 (แสดงจำนวนที่เก็บได้จริง):


,CODE7,รายการ,GROUP,เก็บมาได้,กำหนดให้เก็บ,รายละเอียด
0,1111001,ข้าวสารเจ้า,กระบี่,10,12,2แหล่ง (1หอมมะลิแบบตัก/2หอมมะลิแบบถุง/1ขาวเสาไ...
1,1111001,ข้าวสารเจ้า,สงขลา,9,12,2แหล่ง (1หอมมะลิแบบตัก/2หอมมะลิแบบถุง/1ขาวเสาไ...
2,1111001,ข้าวสารเจ้า,นครศรีธรรมราช,10,12,2แหล่ง (1หอมมะลิแบบตัก/2หอมมะลิแบบถุง/1ขาวเสาไ...
3,1111001,ข้าวสารเจ้า,พิษณุโลก,8,12,2แหล่ง (1หอมมะลิแบบตัก/2หอมมะลิแบบถุง/1ขาวเสาไ...
4,1111001,ข้าวสารเจ้า,ตาก,8,12,2แหล่ง (1หอมมะลิแบบตัก/2หอมมะลิแบบถุง/1ขาวเสาไ...


In [126]:
import xlsxwriter

# ==========================================
# 1. ตั้งชื่อไฟล์ตาม Text
# ==========================================
if text == 'g':
    file_name_export = "การเก็บวิเคราะห์การเก็บ โดยละเอียด ชุดทั่วไป.xlsx"
    print(">>> Mode: General Group (ชุดทั่วไป)")
elif text == 'u':
    file_name_export = "การเก็บวิเคราะห์การเก็บ โดยละเอียด ชุดนอกเขต.xlsx"
    print(">>> Mode: Urban/Rural Group (ชุดนอกเขต)")
else:
    file_name_export = "Analysis_Unknown.xlsx"

# ==========================================
# 3. Export & Format (ทำให้สวยงาม)
# ==========================================
print(f"กำลังบันทึกไฟล์: {file_name_export} ...")
import pandas as pd
import numpy as np
import xlsxwriter
from xlsxwriter.utility import xl_col_to_name

# ... (ส่วน Logic การเตรียมข้อมูล 2.1 - 2.4 คงเดิมตาม Code ก่อนหน้า) ...

# ==========================================
# 3. Export & Format (ฉบับจัดกลาง + บีบคอลัมน์ + เทียบเป้าหมาย)
# ==========================================
print(f"กำลังบันทึกไฟล์: {file_name_export} ...")

with pd.ExcelWriter(file_name_export, engine='xlsxwriter') as writer:
    
    def format_sheet(df, sheet_name, is_matrix=False):
        df.to_excel(writer, sheet_name=sheet_name, index=False)
        workbook  = writer.book
        worksheet = writer.sheets[sheet_name]
        
        # --- Styles Definition ---
        # 1. หัวตาราง (Header): สีน้ำเงิน, ตัวขาว, ตรงกลาง, ตัดคำ
        header_fmt = workbook.add_format({
            'bold': True, 'text_wrap': True, 'align': 'center', 'valign': 'vcenter',
            'fg_color': '#2F75B5', 'font_color': 'white', 'border': 1
        })
        
        # 2. ตัวเลข/รหัส (Numeric): อยู่ตรงกลาง
        center_fmt = workbook.add_format({'align': 'center', 'valign': 'vcenter', 'border': 1})
        
        # 3. ข้อความยาว (Text): ตัดคำ, ชิดซ้าย
        text_wrap_fmt = workbook.add_format({'text_wrap': True, 'valign': 'top', 'border': 1})
        
        # 4. สีแจ้งเตือน (Conditional Formatting)
        red_fmt = workbook.add_format({'bg_color': '#FFC7CE', 'font_color': '#9C0006'})   # ขาด (0)
        green_fmt = workbook.add_format({'bg_color': '#C6EFCE', 'font_color': '#006100'}) # ครบตามเป้า (>= กำหนดให้เก็บ)
        yellow_fmt = workbook.add_format({'bg_color': '#FFEB9C', 'font_color': '#9C5700'}) # มีของแต่ไม่ครบ (< กำหนดให้เก็บ)

        # --- Write Header ---
        for col_num, value in enumerate(df.columns.values):
            worksheet.write(0, col_num, value, header_fmt)
            
        # --- Set Column Width & Format ---
        if is_matrix:
            # === [Sheet 1: Matrix Check] ===
            
            # Col 0: CODE7 (กว้าง 10, กลาง)
            worksheet.set_column(0, 0, 10, center_fmt)
            # Col 1: รายการ (กว้าง 25, ตัดคำ)
            worksheet.set_column(1, 1, 25, text_wrap_fmt)
            # Col 2: กำหนดให้เก็บ (กว้าง 8, กลาง) -> คอลัมน์ C
            worksheet.set_column(2, 2, 8, center_fmt)
            # Col 3: รายละเอียด (กว้าง 30, ตัดคำ)
            worksheet.set_column(3, 3, 30, text_wrap_fmt)
            
            # ** Col 4 ถึงจบ: จังหวัด (บีบให้แคบ กว้าง 6, กลาง) **
            last_col = len(df.columns) - 1
            if last_col >= 4:
                worksheet.set_column(4, last_col, 6, center_fmt)
            
            # Freeze Panes (ตรึงหลังรายละเอียด)
            worksheet.freeze_panes(1, 4)
            
            # --- Conditional Formatting (เทียบกับเป้าหมาย) ---
            # ข้อมูลเริ่มที่ E2 (Row 1, Col 4)
            # เป้าหมายอยู่ที่ C2 (Row 1, Col 2)
            
            # 1. สีแดง: ถ้าเท่ากับ 0
            worksheet.conditional_format(1, 4, len(df), last_col, {
                'type': 'cell', 'criteria': '=', 'value': 0, 'format': red_fmt
            })
            
            # 2. สีเขียว: ถ้า >= คอลัมน์ "กำหนดให้เก็บ" (Column C)
            # สูตร: =E2>=$C2 (เทียบค่าตัวเองกับคอลัมน์ C ในแถวเดียวกัน)
            # ต้องใช้ xl_col_to_name(4) คือ 'E' เพื่อสร้างสูตรเริ่มที่เซลล์แรกของ data range
            start_col_letter = xl_col_to_name(4) # ได้ 'E'
            formula_green = f'={start_col_letter}2>=$C2' 
            
            worksheet.conditional_format(1, 4, len(df), last_col, {
                'type': 'formula', 'criteria': formula_green, 'format': green_fmt
            })

            # 3. (แถม) สีเหลือง: ถ้า > 0 แต่น้อยกว่าเป้าหมาย
            formula_yellow = f'=AND({start_col_letter}2>0, {start_col_letter}2<$C2)'
            worksheet.conditional_format(1, 4, len(df), last_col, {
                'type': 'formula', 'criteria': formula_yellow, 'format': yellow_fmt
            })

        else:
            # === [Sheet 2 & 3: Data List] ===
            for i, col_name in enumerate(df.columns):
                width = 15
                fmt = text_wrap_fmt
                
                if col_name in ['CODE7', 'เก็บมาได้', 'กำหนดให้เก็บ']:
                    width = 10; fmt = center_fmt
                elif col_name == 'รายการ':
                    width = 30
                elif col_name == 'รายละเอียด':
                    width = 40
                elif col_name == 'GROUP':
                    width = 20
                
                worksheet.set_column(i, i, width, fmt)
            
            # Freeze Panes
            freeze_idx = 3 # ตรึงหลัง GROUP
            worksheet.freeze_panes(1, freeze_idx)

    # --- Execute ---
    format_sheet(save1, 'Matrix_Check', is_matrix=True)
    format_sheet(save2, 'All_Data_List', is_matrix=False)
    format_sheet(save3, 'Missing_Only', is_matrix=False)

print(f"✅ เสร็จสิ้น! บันทึกไฟล์เรียบร้อยที่: {file_name_export}")


>>> Mode: Urban/Rural Group (ชุดนอกเขต)
กำลังบันทึกไฟล์: การเก็บวิเคราะห์การเก็บ โดยละเอียด ชุดนอกเขต.xlsx ...
กำลังบันทึกไฟล์: การเก็บวิเคราะห์การเก็บ โดยละเอียด ชุดนอกเขต.xlsx ...
✅ เสร็จสิ้น! บันทึกไฟล์เรียบร้อยที่: การเก็บวิเคราะห์การเก็บ โดยละเอียด ชุดนอกเขต.xlsx


In [127]:
zeros = save3[save3['GROUP']=='ภูเก็ต'].copy()
zeros.rename(columns={'GROUP': 'PROVINCE_NAME','เก็บมาได้':'จังหวัดเก็บ','รายการ':'DESCRIPTION'}, inplace=True)
zeros 

,CODE7,DESCRIPTION,PROVINCE_NAME,จังหวัดเก็บ,กำหนดให้เก็บ,รายละเอียด


In [128]:
print(df_actual.columns)

useCase =df_actual[['ชุด', 'TYPE', 'MONTH', 'YEAR',  'PROVINCE_NAME', 'COMMODITY_CODE', 'DILLER_CODE','DESCRIPTION', 'SHOP_NAME', 'AVG', 'LINK', 'COMPARE_PRICE', 'LAST_PRICE', 'REL',
'COMM1', 'COMM2' ]].copy()

useCase = useCase[useCase['PROVINCE_NAME'] =='ภูเก็ต']
zeros = save3[save3['GROUP']=='ภูเก็ต'].copy()
zeros.rename(columns={'GROUP': 'PROVINCE_NAME','เก็บมาได้':'จังหวัดเก็บ','รายการ':'DESCRIPTION'}, inplace=True)
zeros 

useCase 

# จัด Format รหัส
useCase["COMMODITY_CODE"] = useCase["COMMODITY_CODE"].astype(str).str.zfill(16)
useCase["DILLER_CODE"]    = useCase["DILLER_CODE"].astype(str).str.zfill(10)
useCase["CODE_7_DIGITS"]  = useCase["COMMODITY_CODE"].astype(str).str[:7]
useCase["CODE7"] = useCase["COMMODITY_CODE"].str[0:7]
useCase["CODE9"] = useCase["COMMODITY_CODE"].str[7:16]


# แปลงเป็น str และแทนค่าว่างด้วย '' (สตริงเปล่า) หรือ '0' ตามความเหมาะสม
col1 = useCase['COMMODITY_CODE'].fillna('').astype(str)
col2 = useCase['DILLER_CODE'].fillna('').astype(str)
useCase['ComKeyBasic'] = col1 + col2

# แปลงเป็น str และแทนค่าว่างด้วย '' (สตริงเปล่า) หรือ '0' ตามความเหมาะสม
col1 = useCase['COMMODITY_CODE'].fillna('').astype(str)
col2 = useCase['DILLER_CODE'].fillna('').astype(str)
col3 = useCase['AVG'].fillna(0).astype(str) 
col4 = useCase['COMPARE_PRICE'].fillna(0).astype(str)
useCase['ComKeyAdv'] = col1 + col2 + col3 + col4

# Merge Real_master (RM)
RM = Real_master[['รหัส', 'รายการ','ผู้ดูแล','กำหนดให้เก็บ','รายละเอียด','เวลาเก็บ']].rename(columns={'รหัส': 'CODE_7_DIGITS'})
RM["CODE_7_DIGITS"] = RM["CODE_7_DIGITS"].astype(str).str.zfill(7)
useCase = useCase.merge(RM, on=['CODE_7_DIGITS'], how='left')

# กรอง User

# นับตัวซ้ำ
useCase['นับตัวซ้ำ'] = useCase.groupby(['DILLER_CODE', 'COMMODITY_CODE'])['DILLER_CODE'].transform('count')



def classify_shop_type(x):
    if not isinstance(x, str): return "ร้านอื่นๆ"
    s = x 
    if any(k in s for k in ["โลตัส", "tesco", "Tesco", "โลตัล", "Lotus"]): return "Tesco Lotus"
    if any(k in s for k in ["บิ๊กซี", "บิกซี", "Big C", "Big-C"]): return "Big C"
    if any(k in s for k in ["อีเลฟเว่น", "ELEVEN", "7 - 11", "7-11", "7-ELEVEN", "เซเว่น", "7-Eleven", "seven", "SEVEN", "Eleven", "เซ่เว่น"]): return "Seven Eleven"
    if any(k in s for k in ["CJ", "ซี เจ", "ซีเจ", "Cj Express", "Cj MORE"]): return "CJ EXPRESS"
    if any(k in s for k in ["ท็อปส์ เดลี่", "ท็อปส์ มาร์เก็ต", "ท็อปส์มาร์เก็ต", "Tops", "ห้างท็อปส์", "TOPS"]): return "Tops"
    if any(k in s for k in ["Maxvalue", "แม็กซ์แวลู", "Max Valu"]): return "Maxvalue"
    # if "เดอะมอลล์" in s: return "The Mall"
    # if "แม็คโค" in s: return "Makro"
    if any(k in s for k in ["แม็คโค", "เเม็คโค"]): return "Makro"
    if any(k in s for k in ["กูรเมต์", "กูร์เมต์","กูร์เมต์","กรูเม่"]): return "Gourmet Market"
    if any(k in s for k in ["เดอะมอลล์"]): return "The Mall"
    if any(k in s for k in ["เซนทรัล", "CENTRAL", "เซ็นทรัล"]): return "Central"
    if any(k in s for k in ["วัตสัน", "Watson"]): return "Watson"
    if any(k in s for k in ["เฟรชมาร์ท", "Freshmat"]): return "CP Freshmart"
    if "โฮมโปร" in s: return "HomePro"
    if "ร้านเอเซ่เว่น" in s: return "ร้านอื่นๆ"
    return "ร้านอื่นๆ"

useCase['ประเภทร้านค้า'] = useCase['SHOP_NAME'].apply(classify_shop_type)

# แก้ไขร้านค้ากรณีพิเศษ
mask_other = useCase['SHOP_NAME'].str.contains("ร้านหน้าโลตัส|Watson|วัตสัน|ปลาผา|WASH|Amazon|เอเมซอน|คาเฟ่|Cafe|ศูนย์อาหาร|อาหาร|ร้านยา|ขายยา|ข้าว|DRUG", case=False, na=False)
useCase.loc[mask_other, 'ประเภทร้านค้า'] = "ร้านอื่นๆ"
useCase.loc[useCase['SHOP_NAME'].str.contains("โฮมโปร", case=False, na=False), 'ประเภทร้านค้า'] = "HomePro"

# แก้ไขตามรหัส
useCase.loc[useCase['CODE_7_DIGITS'].astype(str).str.startswith("4111", na=False), 'ประเภทร้านค้า'] = "ร้านอื่นๆ"
useCase.loc[useCase['CODE_7_DIGITS'].astype(str).isin(["1250001", "1250002", "1250008", "1160015"]), 'ประเภทร้านค้า'] = "ร้านอื่นๆ"
useCase.loc[useCase['SHOP_NAME'] == 'ท็อปน์ (เซ็นทรัล นนทบุรี)', 'ประเภทร้านค้า'] = "Tops"
useCase.loc[useCase['SHOP_NAME'] == 'ร้านเอเซ่เว่น', 'ประเภทร้านค้า'] = "ร้านอื่นๆ"

# ..ห้างบิ๊กซีมินิ..... บิ๊กซีมาร์เก็ต และโลตัสเอ็กเพรส ไม่สามารถใช้ราคากลางได้
# 1. Big C Mini
mask_bigc_mini = (useCase['ประเภทร้านค้า'] == 'Big C') & (useCase['SHOP_NAME'].str.contains('มินิ|Mini', case=False, na=False))
useCase.loc[mask_bigc_mini, 'ประเภทร้านค้า'] = 'BigC_mini'
# # 2. Big C Market
# mask_bigc_mkt = (useCase['ประเภทร้านค้า'] == 'Big C') & (useCase['SHOP_NAME'].str.contains('มาร์เก็ต|มาเก็ต|เกต|มาเกต|Market', case=False, na=False))
# useCase.loc[mask_bigc_mkt, 'ประเภทร้านค้า'] = 'BigC_market'
# 3. Lotus Express
mask_lotus_exp = (useCase['ประเภทร้านค้า'] == 'Tesco Lotus') & (useCase['SHOP_NAME'].str.contains('เฟรซ|เฟรช|โก|exp|gofresh', case=False, na=False))
useCase.loc[mask_lotus_exp, 'ประเภทร้านค้า'] = 'Lotus_gofresh'
mask_lotus_exp = (useCase['ประเภทร้านค้า'] == 'Lotus_gofresh') & (useCase['SHOP_NAME'].str.contains('โก้', case=False, na=False))
useCase.loc[mask_lotus_exp, 'ประเภทร้านค้า'] = 'Tesco Lotus'

mask_Tops_day = (useCase['ประเภทร้านค้า'] == 'Tops') & (useCase['SHOP_NAME'].str.contains('เดลี่', case=False, na=False))
useCase.loc[mask_Tops_day, 'ประเภทร้านค้า'] = 'Tops_daily'



# คำนวณขนาดและจำนวน
useCase['ขนาดร้านค้า'] = useCase['ประเภทร้านค้า'].apply(lambda x: 'ร้านขนาดเล็ก' if x == 'ร้านอื่นๆ' else 'ร้านขนาดใหญ่')
for col in ['ขนาดร้านค้า', 'ประเภทร้านค้า']:
    useCase.insert(useCase.columns.get_loc(col)+1, f'จำนวนร้าน({col})', useCase.groupby(['DESCRIPTION', col])[col].transform('count'))
useCase.insert(useCase.columns.get_loc('DESCRIPTION')+1, 'จำนวนสินค้า', useCase.groupby(['DESCRIPTION'])['DESCRIPTION'].transform('count')) 



useCase['จังหวัดเก็บ'] = useCase.groupby(['CODE7','PROVINCE_NAME'])['PROVINCE_NAME'].transform('count')
useCase = pd.concat([useCase, zeros], ignore_index=True)



useCase = useCase.sort_values(by=['CODE7', 'DESCRIPTION', 'ประเภทร้านค้า', 'AVG'], ascending=[True, True, True, False])

myRM = RM[['เวลาเก็บ','CODE_7_DIGITS']].copy()
myRM.rename(columns={'CODE_7_DIGITS': 'CODE7'}, inplace=True)
useCase['เวลาเก็บ'] = useCase['CODE7'].map(myRM.set_index('CODE7')['เวลาเก็บ'])


print(useCase.columns)
useCase = useCase[['ชุด','เวลาเก็บ','MONTH', 'YEAR', 'PROVINCE_NAME','CODE7',  'รายการ', 'COMMODITY_CODE',
       'DILLER_CODE', 'DESCRIPTION', 'SHOP_NAME', 'ประเภทร้านค้า',  'AVG', 'LINK',
       'COMPARE_PRICE', 'LAST_PRICE', 'REL', 'COMM1', 'COMM2', 
         'จังหวัดเก็บ','กำหนดให้เก็บ','รายละเอียด'
       ]]



useCase.to_excel('useCase_puket.xlsx',  index=False)

Index(['ชุด', 'TYPE', 'COMMODITY_CODE', 'MONTH', 'YEAR', 'DILLER_CODE',
       'GROUP_KEY', 'TYPE_CODE', 'AVG', 'C_FLAG', 'G_FLAG', 'L_FLAG', 'N_FLAG',
       'R_FLAG', 'NR_FLAG', 'LINK', 'COMPARE_PRICE', 'LAST_PRICE', 'REL',
       'COMM1', 'COMM2', 'P_FLAG', 'USERID', 'SHOP_NAME', 'PROVINCE_NAME',
       'DESCRIPTION', 'WEEK1', 'WEEK2', 'WEEK3', 'WEEK4', 'WEEK5'],
      dtype='object')
Index(['ชุด', 'TYPE', 'MONTH', 'YEAR', 'PROVINCE_NAME', 'COMMODITY_CODE',
       'DILLER_CODE', 'DESCRIPTION', 'จำนวนสินค้า', 'SHOP_NAME', 'AVG', 'LINK',
       'COMPARE_PRICE', 'LAST_PRICE', 'REL', 'COMM1', 'COMM2', 'CODE_7_DIGITS',
       'CODE7', 'CODE9', 'ComKeyBasic', 'ComKeyAdv', 'รายการ', 'ผู้ดูแล',
       'กำหนดให้เก็บ', 'รายละเอียด', 'เวลาเก็บ', 'นับตัวซ้ำ', 'ประเภทร้านค้า',
       'จำนวนร้าน(ประเภทร้านค้า)', 'ขนาดร้านค้า', 'จำนวนร้าน(ขนาดร้านค้า)',
       'จังหวัดเก็บ'],
      dtype='object')


In [129]:
import pandas as pd

# ... (ส่วนการเตรียมข้อมูล useCase ของคุณ) ...

file_name = 'useCase_puket_grouped.xlsx'
writer = pd.ExcelWriter(file_name, engine='xlsxwriter')
useCase.to_excel(writer, sheet_name='Phuket_Data', index=False)

workbook  = writer.book
worksheet = writer.sheets['Phuket_Data']

# --- 1. นิยาม Format ---

# Format พื้นฐานสำหรับเนื้อหา
base_format = {
    'valign': 'vcenter',
    'align': 'left',
    'font_name': 'Sarabun',
    'border': 1,
    'border_color': '#D3D3D3' # เส้นขอบบางๆ สีเทาอ่อน
}

# Format สำหรับแถวที่เป็นจุดเริ่มต้นของ CODE7 ใหม่ (มีเส้นทึบด้านบน)
group_divider_format = workbook.add_format({
    **base_format,
    'top': 2,           # เส้นขอบบนหนา (ระดับ 2)
    'top_color': 'black' # สีเส้นขอบบน
})

# Format สำหรับแถวปกติ
normal_row_format = workbook.add_format(base_format)

# Format หัวตาราง
header_format = workbook.add_format({
    'bold': True,
    'align': 'center',
    'valign': 'vcenter',
    'fg_color': '#1F4E78',
    'font_color': 'white',
    'border': 1
})

# --- 2. การตกแต่งและเขียนเส้นคั่นกลุ่ม ---

num_rows = len(useCase)
num_cols = len(useCase.columns)
code7_col_idx = useCase.columns.get_loc('CODE7') # หาตำแหน่งคอลัมน์ CODE7

# วนลูปตรวจสอบข้อมูลแต่ละแถว
for row_idx in range(num_rows):
    # row_idx ใน dataframe จะตรงกับ excel row_idx + 1 (เพราะมี header)
    current_code = useCase.iloc[row_idx, code7_col_idx]
    
    # เช็คว่าแถวนี้คือแถวแรกของกลุ่มใหม่หรือไม่
    is_new_group = False
    if row_idx == 0:
        is_new_group = True # แถวแรกสุดเป็นกลุ่มใหม่เสมอ
    else:
        prev_code = useCase.iloc[row_idx - 1, code7_col_idx]
        if current_code != prev_code:
            is_new_group = True
            
    # เลือก format ตามความเหมาะสม
    fmt = group_divider_format if is_new_group else normal_row_format
    
    # เขียนข้อมูลลงในแถวนั้นๆ พร้อมใส่ Format
    for col_idx in range(num_cols):
        val = useCase.iloc[row_idx, col_idx]
        # จัดการค่า NaN เพื่อไม่ให้เขียนคำว่า nan ลงใน Excel
        if pd.isna(val): val = ""
        worksheet.write(row_idx + 1, col_idx, val, fmt)

# ตกแต่งหัวตารางซ้ำอีกครั้ง
for col_num, value in enumerate(useCase.columns.values):
    worksheet.write(0, col_num, value, header_format)

# ตั้งความกว้างคอลัมน์
worksheet.set_column('F:F', 12, None) # คอลัมน์ CODE7
worksheet.set_column('I:J', 35, None) # ชื่อสินค้า/ร้านค้า
worksheet.set_column('A:Z', 15)       # คอลัมน์อื่นๆ ทั่วไป

# ตรึงแนว
worksheet.freeze_panes(1, 0)

writer.close()
print(f"จัดทำไฟล์พร้อมเส้นคั่นกลุ่มเรียบร้อย: {file_name}")

จัดทำไฟล์พร้อมเส้นคั่นกลุ่มเรียบร้อย: useCase_puket_grouped.xlsx


In [130]:
# pd.read_excel('การเก็บวิเคราะห์การเก็บ โดยละเอียด ชุดทั่วไป.xlsx')
# วิธี Save เป็นไฟล์ JSON แบบอ่านไทยได้

if session =='g':
    save1.to_json('my_data.json', 
            orient='records', 
            force_ascii=False, 
            lines=False, 
            indent=4)

    print("Save ไฟล์เรียบร้อยแล้ว!")

if session =='u':
    save1.to_json('my_data2.json', 
            orient='records', 
            force_ascii=False, 
            lines=False, 
            indent=4)

    print("Save ไฟล์เรียบร้อยแล้ว!")



ปปป

Save ไฟล์เรียบร้อยแล้ว!


NameError: name 'ปปป' is not defined

In [ ]:
myRM = RM[['เวลาเก็บ','CODE_7_DIGITS']].copy()
myRM.rename(columns={'CODE_7_DIGITS': 'CODE7'}, inplace=True)
myRM
# RM

,เวลาเก็บ,CODE7
0,สัปดาห์,1111001
1,สัปดาห์,1111002
2,NaN,1112101
3,NaN,1112102
4,NaN,1112103
...,...,...
702,เดือน,7100001
703,NaN,7100002
704,เดือน,7210001
705,เดือน,7220001


,CODE7,รายการ,กำหนดให้เก็บ,รายละเอียด,กทม.กลุ่มจตุจักร,กทม.กลุ่มบางกะปิ,กทม.กลุ่มบางบอน,กทม.กลุ่มบางประกอก,กทม.กลุ่มบางแค,กทม.กลุ่มพระโขนง,...,อุดรธานี,อุตรดิตถ์,อุบลราชธานี,เชียงราย,เชียงใหม่,เพชรบุรี,เพชรบูรณ์,เลย,แพร่,แม่ฮ่องสอน
0,1111001,ข้าวสารเจ้า,12,2แหล่ง (1หอมมะลิแบบตัก/2หอมมะลิแบบถุง/1ขาวเสาไ...,12,14,10,13,7,13,...,12,13,11,12,12,12,13,14,12,7
1,1111002,ข้าวสารเหนียว,4,2แหล่ง (1ข้าวเหนียวถัง/1ข้าวเหนียวถุง),4,2,2,4,2,3,...,2,2,2,2,2,2,2,2,2,2
2,1112104,แป้งทอดกรอบ,4,2แหล่ง 2สเปค,4,4,3,4,4,4,...,3,4,4,4,4,4,4,4,3,9
3,1112202,วุ้นเส้น,8,2แหล่ง (2ขนาดเล็ก/2ขนาดใหญ่),4,4,8,3,3,6,...,6,4,8,7,8,8,8,4,8,6
4,1112203,เต้าหู้,6,2แหล่ง (2เต้าหู้หลอด/1เต้าหู้แผ่นแข็ง),5,5,7,5,6,6,...,6,7,6,6,6,6,6,6,6,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
435,6311002,ค่าอาหารสำหรับตักบาตร,2,2แหล่ง 1สเปค,1,0,0,1,0,0,...,1,2,4,4,2,2,2,0,2,2
436,7100001,บุหรี่,8,2แหล่ง (2บุหรี่นอก/2บุหรี่ใน),2,8,4,2,2,2,...,8,6,8,6,6,7,7,9,5,3
437,7210001,เบียร์,8,2แหล่ง (2เบียร์นอก/2เบียร์ใน),6,9,9,8,4,5,...,6,4,6,4,4,4,3,7,3,7
438,7220001,ไวน์,8,2แหล่ง (2ไวน์นอก/2ไวน์ใน),4,5,3,4,3,3,...,4,2,8,6,3,4,10,2,4,2


In [ ]:
# import pandas as pd
# import numpy as np
# import xlsxwriter

# # ... (โค้ดส่วนเตรียมข้อมูล Logic 2.1 - 2.4 เหมือนเดิมด้านบน) ...

# # ==========================================
# # 3. Export & Format (ฉบับจัดกลาง + บีบคอลัมน์)
# # ==========================================
# print(f"กำลังบันทึกไฟล์: {file_name_export} ...")

# with pd.ExcelWriter(file_name_export, engine='xlsxwriter') as writer:
    
#     # --- Function ช่วยจัดรูปแบบ ---
#     def format_sheet(df, sheet_name, is_matrix=False):
#         df.to_excel(writer, sheet_name=sheet_name, index=False)
#         workbook  = writer.book
#         worksheet = writer.sheets[sheet_name]
        
#         # --- Define Styles ---
#         # หัวตาราง: สีน้ำเงินเข้ม, ตัวขาว, อยู่ตรงกลาง, ตัดคำ
#         header_fmt = workbook.add_format({
#             'bold': True, 'text_wrap': True, 'valign': 'vcenter', 'align': 'center',
#             'fg_color': '#2F75B5', 'font_color': 'white', 'border': 1
#         })
        
#         # ข้อความทั่วไป: ตัดคำ (สำหรับชื่อรายการ, รายละเอียด)
#         text_wrap_fmt = workbook.add_format({'text_wrap': True, 'valign': 'top', 'border': 1})
        
#         # ตัวเลข/รหัส: อยู่ตรงกลาง (สำหรับ CODE7, ค่าที่เก็บได้)
#         center_fmt = workbook.add_format({'align': 'center', 'valign': 'vcenter', 'border': 1})
        
#         # Conditional Formats (สำหรับ Matrix)
#         red_fmt = workbook.add_format({'bg_color': '#FFC7CE', 'font_color': '#9C0006'}) # 0
#         green_fmt = workbook.add_format({'bg_color': '#C6EFCE', 'font_color': '#006100'}) # >0

#         # --- 1. เขียน Header ทับเพื่อใส่ Format ---
#         for col_num, value in enumerate(df.columns.values):
#             worksheet.write(0, col_num, value, header_fmt)
            
#         # --- 2. จัดความกว้างและ Format ตามประเภทคอลัมน์ ---
        
#         if is_matrix:
#             # === สำหรับหน้า Matrix Check ===
#             # คอลัมน์ 0 (CODE7): กว้าง 10, ตรงกลาง
#             worksheet.set_column(0, 0, 10, center_fmt)
#             # คอลัมน์ 1 (รายการ): กว้าง 25, ตัดคำ
#             worksheet.set_column(1, 1, 25, text_wrap_fmt)
#             # คอลัมน์ 2 (กำหนดให้เก็บ): กว้าง 10, ตรงกลาง
#             worksheet.set_column(2, 2, 10, center_fmt)
#             # คอลัมน์ 3 (รายละเอียด): กว้าง 30, ตัดคำ
#             worksheet.set_column(3, 3, 30, text_wrap_fmt)
            
#             # **บีบคอลัมน์จังหวัด**: ตั้งแต่คอลัมน์ 4 ไปจนจบ
#             # ให้กว้างแค่ 6 พอ (เพราะใส่แค่เลขหลักเดียว) และจัดกลาง
#             last_col = len(df.columns) - 1
#             if last_col >= 4:
#                 worksheet.set_column(4, last_col, 6, center_fmt)
                
#             # Freeze Panes (ตรึงหลังรายละเอียด)
#             worksheet.freeze_panes(1, 4)
            
#             # Conditional Formatting (สีแดง/เขียว)
#             worksheet.conditional_format(1, 4, len(df), last_col, {
#                 'type': 'cell', 'criteria': '=', 'value': 0, 'format': red_fmt
#             })
#             worksheet.conditional_format(1, 4, len(df), last_col, {
#                 'type': 'cell', 'criteria': '>', 'value': 0, 'format': green_fmt
#             })
            
#         else:
#             # === สำหรับหน้า List (All Data / Missing) ===
#             # Loop เพื่อเช็คชื่อคอลัมน์แล้วตั้งค่า
#             for i, col_name in enumerate(df.columns):
#                 # Default width
#                 width = 15
#                 fmt = text_wrap_fmt
                
#                 if col_name == 'CODE7':
#                     width = 10; fmt = center_fmt
#                 elif col_name == 'รายการ':
#                     width = 30; fmt = text_wrap_fmt
#                 elif col_name == 'GROUP':
#                     width = 20; fmt = text_wrap_fmt
#                 elif col_name in ['เก็บมาได้', 'กำหนดให้เก็บ']:
#                     width = 12; fmt = center_fmt
#                 elif col_name == 'รายละเอียด':
#                     width = 40; fmt = text_wrap_fmt
                
#                 worksheet.set_column(i, i, width, fmt)
            
#             # Freeze Panes (ตรึงหลัง Group)
#             # หา index ของ GROUP เพื่อตรึงให้ถูก
#             try:
#                 freeze_idx = df.columns.get_loc('GROUP') + 1
#             except:
#                 freeze_idx = 3
#             worksheet.freeze_panes(1, freeze_idx)

#     # --- เรียกใช้ฟังก์ชัน ---
#     format_sheet(save1, 'Matrix_Check', is_matrix=True)
#     format_sheet(save2, 'All_Data_List', is_matrix=False)
#     format_sheet(save3, 'Missing_Only', is_matrix=False)

# print(f"✅ เสร็จสิ้น! บันทึกไฟล์เรียบร้อยที่: {file_name_export}")

In [ ]:
unit7 = df['CODE7'].unique()
unitGROUP = df['GROUP'].unique()    
# unitGROUP = unitGROUP[np.isin(unitGROUP, province_list)]


crosstab_result = pd.crosstab(df['CODE7'], df['GROUP'])
save1 = crosstab_result.copy()
save1 = save1.merge(Detail, on=['CODE7'], how='left')
cols_to_move = ['รายการ', 'กำหนดให้เก็บ', 'รายละเอียด']
remaining_cols = [c for c in save1.columns if c != 'CODE7' and c not in cols_to_move]
new_column_order = ['CODE7'] + cols_to_move + remaining_cols
save1 = save1[new_column_order]

selected_columns = ['CODE7', 'รายการ', 'GROUP', 'เก็บมาได้', 'กำหนดให้เก็บ', 'รายละเอียด']  
# # 2. แปลงตาราง Matrix กลับมาเป็นรายการยาวๆ (Unpivot/Stack)
# จะได้ DataFrame ที่มี 3 คอลัมน์: CODE7, GROUP, และ Status (0 หรือ 1)
result_long = crosstab_result.stack().reset_index(name='เก็บมาได้')
save2 = result_long.copy()
save2 = save2.merge(Detail, on=['CODE7'], how='left')
save2 = save2.reindex(columns=selected_columns)
# 3. กรองเอาเฉพาะที่ "ยังไม่มีการเก็บ" (Status == 0)
# ถ้าต้องการดูอันที่มีแล้ว ให้เปลี่ยนเป็น == 1
missing_data = result_long[result_long['เก็บมาได้'] == 0].copy()
save3 = missing_data.copy()
save3 = save3.merge(Detail, on=['CODE7'], how='left')
save3 = save3.reindex(columns=selected_columns)

# # 4. เลือกเฉพาะคอลัมน์ที่ต้องการแสดงผล
# final_result = missing_data[['CODE7', 'GROUP']]

# # จัดเรียงให้สวยงาม (เรียงตามรหัส และ จังหวัด)
# final_result = final_result.sort_values(by=['CODE7', 'GROUP'])

# # แสดงผล
# print(f"จำนวนรายการที่ยังไม่มีการเก็บ: {len(final_result)} รายการ")

# final_result = final_result.reset_index(drop=True)


# final_result
# result_long .to_clipboard()


In [ ]:
save1.columns

save1


,CODE7,รายการ,กำหนดให้เก็บ,รายละเอียด,กระบี่,กำแพงเพชร,ขอนแก่น,จันทบุรี,ชลบุรี,ชัยนาท,...,สุรินทร์,หนองคาย,อำนาจเจริญ,อุดรธานี,อุตรดิตถ์,อุบลราชธานี,เชียงราย,เชียงใหม่,เพชรบุรี,แพร่
0,1111001,ข้าวสารเจ้า,12.0,2ข้าวหอมมะลิถัง 4ข้าวหอมมะลิถุง / 2ข้าวขาวเสาไ...,10,6,9,8,7,8,...,4,6,8,12,9,7,8,8,10,10
1,1111002,ข้าวสารเหนียว,4.0,2ข้าวเหนียวถัง / 2ข้าวเหนียวถุง,2,2,2,2,2,2,...,2,2,2,2,2,2,2,2,2,2
2,1112104,แป้งทอดกรอบ,4.0,2ร้าน 2ตรา,4,4,4,2,2,4,...,4,3,4,2,2,2,4,4,4,4
3,1112202,วุ้นเส้น,8.0,2ขนาดเล็ก 2ขนาดใหญ่,8,4,4,6,4,6,...,6,4,4,6,5,4,8,8,8,4
4,1112203,เต้าหู้,6.0,4 หลอด 2 แผ่นแข็ง,6,6,3,6,4,6,...,6,2,2,6,4,5,6,4,4,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
448,6311002,ค่าอาหารสำหรับตักบาตร,2.0,2แหล่ง,4,0,0,2,2,0,...,4,0,0,0,0,0,4,0,4,0
449,7100001,บุหรี่,8.0,2 ร้าน 2ตรา (บุหรี่นอก บุหรี่ใน),6,6,6,5,5,4,...,5,4,6,4,5,6,4,3,9,6
450,7210001,เบียร์,8.0,2 ร้าน 2ตรา (เบียร์นอก เบียร์ใน),4,6,4,3,4,4,...,4,3,8,4,5,4,4,2,6,3
451,7220001,ไวน์,8.0,2 ร้าน 2ตรา (ไวน์นอก ไวน์ใน),2,2,1,1,4,2,...,2,1,4,0,6,2,5,1,3,4


### ยังไม่เอา

In [ ]:
import pandas as pd

# 1. เตรียมรายการรหัส (Unique) และ จังหวัด (Unique)
unit7 = df['CODE_7_DIGITS'].unique()
unitGROUP = df['GROUP'].unique()

# 2. สร้างตารางจำลอง "ทุกคู่ที่เป็นไปได้" (All Combinations)
# บรรทัดนี้จะจับคู่ทุกรหัส กับ ทุกจังหวัด เตรียมไว้ก่อน
all_combinations = pd.MultiIndex.from_product(
    [unit7, unitGROUP], 
    names=['CODE_7_DIGITS', 'GROUP']
)

# 3. สร้าง Index จากข้อมูลจริงที่มีอยู่ (Actual Data)
# เพื่อเอาไว้เทียบว่าอันไหนมีแล้ว
actual_combinations = df.set_index(['CODE_7_DIGITS', 'GROUP']).index

# 4. หา "ส่วนต่าง" (Difference)
# คือเอา (ทั้งหมดที่เป็นไปได้) - (สิ่งที่มีอยู่จริง) = สิ่งที่ขาดหายไป (Missing)
missing_combinations = all_combinations.difference(actual_combinations)

# 5. แปลงผลลัพธ์กลับเป็น DataFrame เพื่อนำไปใช้ต่อ
missing_df = missing_combinations.to_frame(index=False).reset_index(drop=True)

# จัดเรียงให้สวยงาม
missing_df = missing_df.sort_values(by=['CODE_7_DIGITS', 'GROUP'])

print(f"พบรายการที่ยังไม่มีการเก็บจำนวน: {len(missing_df)} รายการ")
display(missing_df.head(10)) # ดูตัวอย่าง 10 แถวแรก

พบรายการที่ยังไม่มีการเก็บจำนวน: 17335 รายการ


,CODE_7_DIGITS,GROUP
0,1111001,กทม.กลุ่มประตูน้ำ
1,1111001,กทม.กลุ่มเขตพิเศษ
2,1111001,กทม.ซีเจมอร์oneprice
3,1111001,กำแพงเพชร
4,1111001,ชัยนาท
5,1111001,ชัยภูมิ
6,1111001,ชุมพร
7,1111001,ตราด
8,1111001,บุรีรัมย์
9,1111001,ปัตตานี


In [ ]:
text = 'g'

name_series = 'แบ่งกลุ่มฐานลูกจ้าง_15-16 มค 69.xlsx'

dG = pd.read_excel(name_series,skiprows=1, engine='calamine')
# dG = dG[['section','กลุ่มและจังหวัด','สรุปรหัส']]

dG['สรุปรหัส'] = pd.to_numeric(dG['สรุปรหัส'], errors='coerce').fillna(0).astype(int)
dG = dG[dG['สรุปรหัส'] != 0]
dG['สรุปรหัส'] = dG['สรุปรหัส'].astype(str).str.zfill(7)
dG = dG.reset_index(drop=True)
dCODE = dG['สรุปรหัส'].copy()

dG = pd.read_excel(name_series, skiprows=1, engine='calamine')
dG = dG[['ชุด','กลุ่มและจังหวัด','แบ่งกลุ่ม','สรุปรหัส','สรุปชื่อไฟล์']]
# display(dG.head())
if text == 'g':
    dG = dG[dG['ชุด'] == 'ชุดทั่วไป']
if text == 'u':
    dG = dG[dG['ชุด'] == 'ชุดนอกเขต']
dG = dG[['ชุด','กลุ่มและจังหวัด','แบ่งกลุ่ม','สรุปรหัส','สรุปชื่อไฟล์']]
# # จัดการรหัส
dG['สรุปรหัส'] = pd.to_numeric(dG['สรุปรหัส'], errors='coerce').fillna(0).astype(int)
dG = dG[dG['สรุปรหัส'] != 0]
dG['สรุปรหัส'] = dG['สรุปรหัส'].astype(str).str.zfill(7)
dG = dG.reset_index(drop=True)

# --- แก้ไขตรงนี้ครับ ---
# ใช้ .str.replace เพื่อลบ \n ที่ซ่อนอยู่ในข้อความ
dG['สรุปชื่อไฟล์'] = dG['สรุปชื่อไฟล์'].astype(str).str.replace('\n', '', regex=False)

# แยกข้อความด้วย "_" แล้วเอาตัวแรก (index 0) เก็บไว้ในคอลัมน์ใหม่ ชื่อ 'Group'
# dG['แบ่งกลุ่ม_'] = dG['สรุปชื่อไฟล์'].str.split('_').str[0]

dG['ชื่อไฟล์_ตัดGroup'] = dG['สรุปชื่อไฟล์'].str.split('_', n=1).str[1]
dG['กลุ่มและจังหวัด'] = dG['กลุ่มและจังหวัด'].astype(str).str.replace('\n', '', regex=False)

# หรือถ้าจะใช้แบบเดิม ต้องเพิ่ม regex=True เข้าไปครับ
# dG['สรุปชื่อไฟล์'] = dG['สรุปชื่อไฟล์'].replace('\n', '', regex=True)
dCODEmap = dG.reset_index(drop=True).copy()

dG = pd.read_excel(name_series,skiprows=1, engine='calamine')
dG = dG[['section','กลุ่มและจังหวัด','แบ่งกลุ่ม','ชุด']]

# dG['แบ่งกลุ่ม'] = pd.to_numeric(dG['แบ่งกลุ่ม'], errors='coerce').fillna(0).astype(int)
dG = dG.dropna(subset=['section'])
dG = dG.reset_index(drop=True)
dGroup = dG.copy()

dGroup = dGroup[['กลุ่มและจังหวัด','แบ่งกลุ่ม','ชุด']]
if text == 'g':
    dGroup = dGroup[dGroup['ชุด']=='ชุดทั่วไป']
elif text == 'u':
    dGroup = dGroup[dGroup['ชุด']=='ชุดนอกเขต']
dGroup = dGroup[['กลุ่มและจังหวัด','แบ่งกลุ่ม']]    
dGroup.reset_index(drop=True)
dGroup['กลุ่มและจังหวัด'] = dGroup['กลุ่มและจังหวัด'].astype(str).str.replace('\n', '', regex=False)
# dGroup
# dG

In [ ]:
dGroup
dCODEmap

dCODE


0      1111001
1      1131001
2      1142101
3      1121107
4      1141115
        ...   
100    6110007
101    6150003
102    5120002
103    5230004
104    4221004
Name: สรุปรหัส, Length: 105, dtype: object

In [ ]:
mg = mg_.copy()
mg.insert(0, 'ชุด', 'อ.เมือง')
mu = mu_.copy()
mu.insert(0, 'ชุด', 'นอกเขต')
wg = wg_.copy()
wg.insert(0, 'ชุด', 'อ.เมือง')
wu = wu_.copy()
wu.insert(0, 'ชุด', 'นอกเขต')

# --- ต่อจาก Code ของคุณ ---

# 1. เลือกว่าจะใช้ชุดข้อมูลไหนตรวจสอบ (ตามตัวแปร text)
if text == 'g':
    # รวมข้อมูล General (รายเดือน + รายสัปดาห์)
    df_actual = pd.concat([mg, wg], ignore_index=True)
    target_set_name = 'ชุดทั่วไป'
elif text == 'u':
    # รวมข้อมูล Rural/Urban (รายเดือน + รายสัปดาห์)
    df_actual = pd.concat([mu, wu], ignore_index=True)
    target_set_name = 'ชุดนอกเขต'
else:
    raise ValueError("ตัวแปร text ต้องเป็น 'g' หรือ 'u' เท่านั้น")


In [ ]:

# 2. เตรียม Master Data (List ที่ต้องมี)
# ดึงจาก dCODEmap ที่คุณทำไว้แล้ว กรองเฉพาะชุดที่ต้องการ
df_master = dCODEmap[dCODEmap['ชุด'] == target_set_name].copy()

# สร้าง Key สำหรับตรวจสอบ: "ชื่อจังหวัด_รหัสสินค้า"
# ต้องแน่ใจว่า format รหัสเป็น 7 หลักเหมือนกัน
df_master['check_key'] = df_master['กลุ่มและจังหวัด'].astype(str).str.strip() + '_' + df_master['สรุปรหัส'].astype(str).str.strip()

df_master

,ชุด,กลุ่มและจังหวัด,แบ่งกลุ่ม,สรุปรหัส,สรุปชื่อไฟล์,ชื่อไฟล์_ตัดGroup,check_key
0,ชุดทั่วไป,กทม.กลุ่มจตุจักร,central1,1111001,กลุ่มจตุจักร_ชุดทั่วไป_1111001_ข้าวสารเจ้า[ฐาน1],ชุดทั่วไป_1111001_ข้าวสารเจ้า[ฐาน1],กทม.กลุ่มจตุจักร_1111001
1,ชุดทั่วไป,กทม.กลุ่มสะพานใหม่,central1,1131001,กลุ่มสะพานใหม่_ชุดทั่วไป_1131001_ไข่ไก่[ฐาน1],ชุดทั่วไป_1131001_ไข่ไก่[ฐาน1],กทม.กลุ่มสะพานใหม่_1131001
2,ชุดทั่วไป,กทม.กลุ่มพรานนก,central1,1142101,กลุ่มพรานนก_ชุดทั่วไป_1142101_กล้วยน้ำว้า[ฐาน1],ชุดทั่วไป_1142101_กล้วยน้ำว้า[ฐาน1],กทม.กลุ่มพรานนก_1142101
3,ชุดทั่วไป,กทม.กลุ่มบางบอน,central1,1121107,กลุ่มบางบอน_ชุดทั่วไป_1121107_เนื้อสุกรบด[ฐาน1],ชุดทั่วไป_1121107_เนื้อสุกรบด[ฐาน1],กทม.กลุ่มบางบอน_1121107
4,ชุดทั่วไป,สมุทรปราการ,central1,1141115,สมุทรปราการ_ชุดทั่วไป_1141115_มะนาว[ฐาน1],ชุดทั่วไป_1141115_มะนาว[ฐาน1],สมุทรปราการ_1141115
...,...,...,...,...,...,...,...
63,ชุดทั่วไป,ร้อยเอ็ด,North east,6125003,ร้อยเอ็ด_ชุดทั่วไป_6125003_ค่าอาบน้ำ/ตัดขนสัตว...,ชุดทั่วไป_6125003_ค่าอาบน้ำ/ตัดขนสัตว์[ฐาน3],ร้อยเอ็ด_6125003
64,ชุดทั่วไป,นครพนม,North east,6150001,นครพนม_ชุดทั่วไป_6150001_ค่าเดินทางไปเยี่ยมญาต...,ชุดทั่วไป_6150001_ค่าเดินทางไปเยี่ยมญาติและทำบ...,นครพนม_6150001
65,ชุดทั่วไป,กระบี่,South,6110007,กระบี่_ชุดทั่วไป_6110007_ค่าสมาชิกฟิตเนส[ฐาน3],ชุดทั่วไป_6110007_ค่าสมาชิกฟิตเนส[ฐาน3],กระบี่_6110007
66,ชุดทั่วไป,สุราษฎร์ธานี,South,6150003,สุราษฎร์ธานี_ชุดทั่วไป_6150003_ค่าเช่ารถ (ขับเ...,ชุดทั่วไป_6150003_ค่าเช่ารถ (ขับเอง)[ฐาน3],สุราษฎร์ธานี_6150003


In [ ]:
import pandas as pd

# --- ส่วนของการตั้งค่า (Configuration) ---
text = 'g'  # เลือก 'g' หรือ 'u' ตามต้องการ

# ตั้งชื่อไฟล์
name_series = 'แบ่งกลุ่มฐานลูกจ้าง_15-16 มค 69.xlsx'

# --- 1. อ่านและเตรียมข้อมูล Master (dCODEmap) ตามโค้ดของคุณ ---
dG = pd.read_excel(name_series, skiprows=1, engine='calamine')
dG = dG[['ชุด','กลุ่มและจังหวัด','แบ่งกลุ่ม','สรุปรหัส','สรุปชื่อไฟล์']]

# กรองตาม text ('g' หรือ 'u')
if text == 'g':
    dG = dG[dG['ชุด'] == 'ชุดทั่วไป']
    target_set_name = 'ชุดทั่วไป'
elif text == 'u':
    dG = dG[dG['ชุด'] == 'ชุดนอกเขต']
    target_set_name = 'ชุดนอกเขต'

# จัดการข้อมูล Master
dG['สรุปรหัส'] = pd.to_numeric(dG['สรุปรหัส'], errors='coerce').fillna(0).astype(int)
dG = dG[dG['สรุปรหัส'] != 0]
dG['สรุปรหัส'] = dG['สรุปรหัส'].astype(str).str.zfill(7)
dG['กลุ่มและจังหวัด'] = dG['กลุ่มและจังหวัด'].astype(str).str.replace('\n', '', regex=False).str.strip()
dG['สรุปชื่อไฟล์'] = dG['สรุปชื่อไฟล์'].astype(str).str.replace('\n', '', regex=False)
dG = dG.reset_index(drop=True)
dCODEmap = dG.copy()

# --- 2. เตรียมข้อมูล Actual (สิ่งที่เก็บมาแล้ว) ---
# โหลดข้อมูลจริง (สมมติว่าอ่านไฟล์มาแล้วเป็น mg_, wg_, mu_, wu_)
# mg = mg_.copy() ... (ตามโค้ดของคุณ)

if text == 'g':
    df_actual = pd.concat([mg, wg], ignore_index=True)
elif text == 'u':
    df_actual = pd.concat([mu, wu], ignore_index=True)

# *** (สำคัญ) ระบุชื่อคอลัมน์ จังหวัด และ รหัสสินค้า ในไฟล์ Excel ข้อมูลดิบของคุณ ***
col_prov_actual = 'CWT_NAME'      # <-- เปลี่ยนเป็นชื่อคอลัมน์จังหวัดในไฟล์ cpig/cpiu
col_code_actual = 'PRODUCT_CODE'  # <-- เปลี่ยนเป็นชื่อคอลัมน์รหัสสินค้าในไฟล์ cpig/cpiu

# ตรวจสอบและจัดการข้อมูล Actual
if col_prov_actual in df_actual.columns and col_code_actual in df_actual.columns:
    # Clean Data
    df_actual['clean_code'] = pd.to_numeric(df_actual[col_code_actual], errors='coerce').fillna(0).astype(int).astype(str).str.zfill(7)
    df_actual['clean_prov'] = df_actual[col_prov_actual].astype(str).str.strip()
    
    # สร้าง Key สำหรับเทียบ (จังหวัด + รหัส)
    df_actual['key_check'] = df_actual['clean_prov'] + '_' + df_actual['clean_code']
    dCODEmap['key_check'] = dCODEmap['กลุ่มและจังหวัด'] + '_' + dCODEmap['สรุปรหัส']
    
    # --- 3. หาตัวที่ขาด (Missing) ---
    existing_keys = set(df_actual['key_check'])
    # กรองเอาเฉพาะรายการใน Master ที่ Key ไม่อยู่ใน Actual
    df_missing = dCODEmap[~dCODEmap['key_check'].isin(existing_keys)].copy()
    
    # เรียงข้อมูลเพื่อความสวยงาม
    df_missing = df_missing.sort_values(by=['กลุ่มและจังหวัด', 'สรุปรหัส'])

    # --- 4. จัดรูปแบบใหม่ (Pivot) ตามที่ต้องการ ---
    # Group by จังหวัด แล้วเอา รหัสที่ขาด มารวมเป็น list
    missing_grouped = df_missing.groupby('กลุ่มและจังหวัด')['สรุปรหัส'].apply(list)
    
    # กระจาย list ออกเป็น Columns
    df_output = pd.DataFrame(missing_grouped.tolist(), index=missing_grouped.index)
    
    # เปลี่ยนชื่อ Column เป็น รหัสที่ขาด1, รหัสที่ขาด2, ...
    df_output.columns = [f'รหัสที่ขาด{i+1}' for i in range(df_output.shape[1])]
    
    # Reset Index เพื่อให้จังหวัดกลับมาเป็น Column แรก
    df_output = df_output.reset_index()
    
    # แสดงผล
    print(f"จำนวนจังหวัดที่มีรายการขาด: {len(df_output)}")
    display(df_output.head())
    
    # หากต้องการ Save เป็น Excel
    # df_output.to_excel('missing_codes_matrix.xlsx', index=False)

else:
    print(f"Error: ไม่พบคอลัมน์ '{col_prov_actual}' หรือ '{col_code_actual}' ในไฟล์ข้อมูล")

Error: ไม่พบคอลัมน์ 'CWT_NAME' หรือ 'PRODUCT_CODE' ในไฟล์ข้อมูล


In [ ]:
import pandas as pd
import numpy as np

# --- 1. ตั้งค่าและโหลดไฟล์ ---
text = 'g'  # เลือก 'g' (ชุดทั่วไป/ในเมือง) หรือ 'u' (ชุดนอกเขต/อำเภอ)

# โหลดไฟล์ข้อมูลดิบ (Actual Data)
mg = pd.read_excel('cpig_month_2568-12.xlsx', engine= "calamine")
wg = pd.read_excel('cpig_week_2568-12.xlsx', engine= "calamine")
mu = pd.read_excel('cpiu_month_2568-12.xlsx', engine= "calamine")
wu = pd.read_excel('cpiu_week_2568-12.xlsx', engine= "calamine")

# โหลดไฟล์แม่บท (Master List)
name_series = 'แบ่งกลุ่มฐานลูกจ้าง_15-16 มค 69.xlsx'
dG = pd.read_excel(name_series, skiprows=1)
dG = dG[['ชุด', 'กลุ่มและจังหวัด', 'แบ่งกลุ่ม', 'สรุปรหัส', 'สรุปชื่อไฟล์']]

# --- 2. เตรียมข้อมูลฝั่ง Master (สิ่งที่ต้องมี) ---
# กรองตามชุดที่เลือก
if text == 'g':
    target_set = 'ชุดทั่วไป'
    df_actual = pd.concat([mg, wg], ignore_index=True)
    flag_cols = ['C_FLAG', 'G_FLAG', 'L_FLAG', 'R_FLAG']
    df_actual['TOTAL_FLAGS'] = np.where(df_actual[flag_cols].sum(axis=1) > 0, 'คำนวณ', 'ไม่คำนวณ')
elif text == 'u':
    target_set = 'ชุดนอกเขต'
    df_actual = pd.concat([mu, wu], ignore_index=True)

# Clean Data ฝั่ง Master
dG = dG[dG['ชุด'] == target_set].copy()
dG['สรุปรหัส'] = pd.to_numeric(dG['สรุปรหัส'], errors='coerce').fillna(0).astype(int)
dG = dG[dG['สรุปรหัส'] != 0]
dG['code_master'] = dG['สรุปรหัส'].astype(str).str.zfill(7)
# ลบช่องว่างและ \n ในชื่อจังหวัด
dG['prov_master'] = dG['กลุ่มและจังหวัด'].astype(str).str.replace('\n', '', regex=False).str.strip()

# สร้าง Key สำหรับตรวจสอบ (จังหวัด + รหัส 7 หลัก)
dG['key_check'] = dG['prov_master'] + '_' + dG['code_master']



In [ ]:
df_actual = df_actual[df_actual['TOTAL_FLAGS'] != 'ไม่คำนวณ'].copy()

In [ ]:
# ds_ = pd.read_excel('shop_data_20251117_134054.xlsx',skiprows= 2)
import pandas as pd
import glob
# ค้นหาไฟล์ทั้งหมดที่ขึ้นต้นด้วย shop_data และลงท้ายด้วย .xlsx
files_shop = glob.glob("shop_data*.xlsx")
ds_ = pd.read_excel(files_shop [0], skiprows=2, engine='calamine')
# Dealer Group
ds = ds_[["รหัสร้าน", "กลุ่ม"]].rename(columns={"รหัสร้าน": "DILLER_CODE"})
ds["DILLER_CODE"] = ds["DILLER_CODE"].astype(str).str.zfill(10)
ds["กลุ่ม"] = ds["กลุ่ม"].replace(["-", " "], "", regex=True)
df_actual = df_actual.merge(ds, on=["DILLER_CODE"], how="left")
ds

ValueError: You are trying to merge on int64 and object columns for key 'DILLER_CODE'. If you wish to proceed you should use pd.concat

In [ ]:
df_actual['COMMODITY_CODE'] = pd.to_numeric(df_actual['COMMODITY_CODE'], errors='coerce').fillna(0).astype(int).astype(str).str.zfill(7)
df_actual['CODE7'] = df_actual['COMMODITY_CODE'].str[:7]
uni1 = df_actual['CODE7'].unique()
uni1 = sorted(uni1)  # Sorting the unique values of CODE7
uni1 = [code for code in uni1 if code != "0000000"]  # Removing "0000000" from the list
print(len(uni1))
uni1

466


['1111001',
 '1111002',
 '1112104',
 '1112202',
 '1112203',
 '1112205',
 '1112207',
 '1112210',
 '1121101',
 '1121103',
 '1121104',
 '1121105',
 '1121106',
 '1121107',
 '1121203',
 '1121204',
 '1121205',
 '1121207',
 '1121209',
 '1121210',
 '1121211',
 '1121212',
 '1122102',
 '1123101',
 '1123102',
 '1123103',
 '1123104',
 '1123202',
 '1123203',
 '1123204',
 '1123210',
 '1123301',
 '1123302',
 '1123304',
 '1123305',
 '1123306',
 '1123307',
 '1123309',
 '1123310',
 '1123401',
 '1123402',
 '1123403',
 '1123404',
 '1123406',
 '1123407',
 '1123408',
 '1123501',
 '1123503',
 '1123504',
 '1123505',
 '1131001',
 '1131002',
 '1132001',
 '1132002',
 '1132003',
 '1132004',
 '1132006',
 '1141101',
 '1141102',
 '1141103',
 '1141104',
 '1141105',
 '1141106',
 '1141107',
 '1141108',
 '1141109',
 '1141110',
 '1141111',
 '1141112',
 '1141113',
 '1141114',
 '1141115',
 '1141116',
 '1141117',
 '1141118',
 '1141119',
 '1141121',
 '1141122',
 '1141123',
 '1141126',
 '1141127',
 '1141128',
 '1141129',
 '11

ValueError: You are trying to merge on int64 and object columns for key 'DILLER_CODE'. If you wish to proceed you should use pd.concat

In [ ]:
dG

,ชุด,กลุ่มและจังหวัด,แบ่งกลุ่ม,สรุปรหัส,สรุปชื่อไฟล์,code_master,prov_master,key_check
0,ชุดทั่วไป,กทม.กลุ่มจตุจักร,central1,1111001,กลุ่มจตุจักร_ชุดทั่วไป_1111001_ข้าวสารเจ้า[ฐาน1],1111001,กทม.กลุ่มจตุจักร,กทม.กลุ่มจตุจักร_1111001
1,ชุดทั่วไป,\nกทม.กลุ่มสะพานใหม่,central1,1131001,กลุ่มสะพานใหม่_ชุดทั่วไป_1131001_ไข่ไก่[ฐาน1],1131001,กทม.กลุ่มสะพานใหม่,กทม.กลุ่มสะพานใหม่_1131001
2,ชุดทั่วไป,กทม.กลุ่มพรานนก,central1,1142101,กลุ่มพรานนก_ชุดทั่วไป_1142101_กล้วยน้ำว้า[ฐาน1],1142101,กทม.กลุ่มพรานนก,กทม.กลุ่มพรานนก_1142101
3,ชุดทั่วไป,กทม.กลุ่มบางบอน,central1,1121107,กลุ่มบางบอน_ชุดทั่วไป_1121107_เนื้อสุกรบด[ฐาน1],1121107,กทม.กลุ่มบางบอน,กทม.กลุ่มบางบอน_1121107
4,ชุดทั่วไป,สมุทรปราการ,central1,1141115,สมุทรปราการ_ชุดทั่วไป_1141115_มะนาว[ฐาน1],1141115,สมุทรปราการ,สมุทรปราการ_1141115
...,...,...,...,...,...,...,...,...
96,ชุดทั่วไป,ร้อยเอ็ด,North east,6125003,ร้อยเอ็ด_ชุดทั่วไป_6125003_ค่าอาบน้ำ/ตัดขนสัตว...,6125003,ร้อยเอ็ด,ร้อยเอ็ด_6125003
97,ชุดทั่วไป,นครพนม,North east,6150001,นครพนม_ชุดทั่วไป_6150001_ค่าเดินทางไปเยี่ยมญาต...,6150001,นครพนม,นครพนม_6150001
101,ชุดทั่วไป,กระบี่,South,6110007,กระบี่_ชุดทั่วไป_6110007_ค่าสมาชิกฟิตเนส[ฐาน3],6110007,กระบี่,กระบี่_6110007
102,ชุดทั่วไป,สุราษฎร์ธานี,South,6150003,สุราษฎร์ธานี_ชุดทั่วไป_6150003_ค่าเช่ารถ (ขับเ...,6150003,สุราษฎร์ธานี,สุราษฎร์ธานี_6150003


In [ ]:
def find_missing_codes(d, uni1):
    
    missing_codes = d[~d['code_master'].isin(uni1)].copy()
    return missing_codes

In [ ]:

# --- 3. เตรียมข้อมูลฝั่ง Actual (สิ่งที่มีแล้ว) ---
# กำหนดชื่อคอลัมน์ให้ตรงกับไฟล์จริง
col_prov = 'PROVINCE_NAME'     # ชื่อคอลัมน์จังหวัด
col_code = 'COMMODITY_CODE'    # ชื่อคอลัมน์รหัสสินค้า (16 หลัก)

if col_prov in df_actual.columns and col_code in df_actual.columns:
    # ตัดรหัสสินค้าเอาแค่ 7 หลักแรก เพื่อให้ตรงกับ Master
    df_actual['code_7'] = df_actual[col_code].astype(str).str.slice(0, 7).str.zfill(7)
    
    # Clean ชื่อจังหวัด
    df_actual['prov_clean'] = df_actual[col_prov].astype(str).str.strip()
    
    # สร้าง Key ฝั่งข้อมูลจริง
    df_actual['key_check'] = df_actual['prov_clean'] + '_' + df_actual['code_7']
    
    # --- 4. หาผลต่าง (Missing Items) ---
    existing_keys = set(df_actual['key_check'])
    
    # กรองเอาเฉพาะรายการใน Master ที่ Key ไม่เจอในข้อมูลจริง
    df_missing = dG[~dG['key_check'].isin(existing_keys)].copy()
    
    # --- 5. จัดรูปแบบตาราง (Pivot) ตามที่ต้องการ ---
    if not df_missing.empty:
        # เรียงข้อมูล
        df_missing = df_missing.sort_values(by=['กลุ่มและจังหวัด', 'สรุปรหัส'])
        
        # Group ตามจังหวัด แล้วรวมรหัสที่ขาดเป็น List
        missing_grouped = df_missing.groupby('prov_master')['code_master'].apply(list)
        
        # กระจาย List ออกเป็นหลายคอลัมน์
        max_missing = missing_grouped.apply(len).max()
        col_names = [f'รหัสที่ขาด{i+1}' for i in range(max_missing)]
        
        # สร้าง DataFrame ผลลัพธ์
        df_result = pd.DataFrame(missing_grouped.tolist(), index=missing_grouped.index, columns=col_names)
        
        # Reset Index ให้ชื่อจังหวัดเป็นคอลัมน์แรก
        df_result = df_result.reset_index().rename(columns={'prov_master': 'จังหวัด'})
        
        print(f"พบรายการที่ขาดใน {len(df_result)} กลุ่ม/จังหวัด")
        # แสดงผลลัพธ์
        display(df_result.head())
        
        # หากต้องการ Save ไฟล์
        # df_result.to_excel('missing_codes_result.xlsx', index=False)
        
    else:
        print("สุดยอด! ไม่พบรายการที่ขาด (เก็บครบทุกจังหวัด)")

else:
    print(f"Error: ไม่พบคอลัมน์ '{col_prov}' หรือ '{col_code}' ในไฟล์")
    print("คอลัมน์ที่มี:", df_actual.columns.tolist())

In [ ]:
import pandas as pd

# โหลดข้อมูล
mg = pd.read_excel('cpig_month_2568-12.xlsx')
wg = pd.read_excel('cpig_week_2568-12.xlsx')
mu = pd.read_excel('cpiu_month_2568-12.xlsx')
wu = pd.read_excel('cpiu_week_2568-12.xlsx')
dG = pd.read_excel('แบ่งกลุ่มฐานลูกจ้าง_15-16 มค 69.xlsx', skiprows=1)

# กำหนดเงื่อนไข
text = 'g' # เปลี่ยนเป็น 'u' หากต้องการชุดนอกเขต
target_set = 'ชุดทั่วไป' if text == 'g' else 'ชุดนอกเขต'

# --- 1. เตรียม Master Data ---
master = dG[dG['ชุด'] == target_set].copy()
master['Code'] = pd.to_numeric(master['สรุปรหัส'], errors='coerce').fillna(0).astype(int).astype(str).str.zfill(7)
master['Province'] = master['กลุ่มและจังหวัด'].astype(str).str.replace('\n', '', regex=False).str.strip()
master = master[master['Code'] != '0000000']
master['Key'] = master['Province'] + '_' + master['Code']

# --- 2. เตรียม Actual Data ---
if text == 'g':
    actual = pd.concat([mg, wg], ignore_index=True)
else:
    actual = pd.concat([mu, wu], ignore_index=True)

# แปลงรหัสสินค้า 16 หลัก เป็น 7 หลัก
actual['Code_Full'] = pd.to_numeric(actual['COMMODITY_CODE'], errors='coerce').fillna(0).astype(int).astype(str)
actual['Code_7'] = actual['Code_Full'].str.slice(0, 7).str.zfill(7) # ตัด 7 หลักแรก
actual['Province'] = actual['PROVINCE_NAME'].astype(str).str.strip()
actual['Key'] = actual['Province'] + '_' + actual['Code_7']

# --- 3. หาตัวที่ขาด (Set Difference) ---
missing_keys = set(master['Key']) - set(actual['Key'])
missing_df = master[master['Key'].isin(missing_keys)].copy()

# --- 4. จัดรูปแบบ Wide Format ---
if not missing_df.empty:
    # เรียงข้อมูล
    missing_df = missing_df.sort_values(by=['Province', 'Code'])
    
    # Group รวมรายการที่ขาดเป็น list
    grouped = missing_df.groupby('Province')['Code'].apply(list)
    
    # แตก list ออกเป็น columns
    result_df = pd.DataFrame(grouped.tolist(), index=grouped.index)
    result_df.columns = [f'รหัสที่ขาด{i+1}' for i in range(result_df.shape[1])]
    result_df = result_df.reset_index().rename(columns={'Province': 'จังหวัด'})
    
    print(f"พบรายการที่ขาดใน {len(result_df)} จังหวัด")
    display(result_df.head())
else:
    print("ไม่พบรายการที่ขาด")

พบรายการที่ขาดใน 15 จังหวัด


,จังหวัด,รหัสที่ขาด1
0,กทม.กลุ่มจตุจักร,1111001
1,กทม.กลุ่มบางกะปิ,4210012
2,กทม.กลุ่มบางบอน,1121107
3,กทม.กลุ่มบางประกอก,6110001
4,กทม.กลุ่มบางแค,4221006
